In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "customer360_dev.bronze.subscriptions"
)

silver_df = spark.table(
    "customer360_dev.silver.subscriptions"
)

reject_df = spark.table(
    "customer360_dev.quarantine.subscription_rejects"
)

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows   : {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rejected rows : {reject_count:,}")

Bronze rows   : 1,000,066
Silver rows   : 978,054
Rejected rows : 22,012


Rebuild valid-candidate logic

In [0]:
customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

validated_df = (
    bronze_df
    .withColumn(
        "start_date_cast",
        F.to_date("start_date", "yyyy-MM-dd")
    )
    .withColumn(
        "end_date_cast",
        F.to_date("end_date", "yyyy-MM-dd")
    )
    .withColumn(
        "monthly_fee_cast",
        F.col("monthly_fee").cast("decimal(10,2)")
    )
    .withColumn(
        "plan_code_clean",
        F.upper(F.trim("plan_code"))
    )
    .withColumn(
        "subscription_status_clean",
        F.upper(F.trim("subscription_status"))
    )
    .withColumn(
        "billing_cycle_clean",
        F.upper(F.trim("billing_cycle"))
    )
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
    .withColumn(
        "dq_missing_subscription_id",
        F.col("subscription_id").isNull()
        | (F.trim(F.col("subscription_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("subscription_status_clean").isin(
            "ACTIVE",
            "PAUSED",
            "CANCELLED",
            "EXPIRED"
        )
    )
    .withColumn(
        "dq_invalid_plan",
        ~F.col("plan_code_clean").isin(
            "BASIC",
            "STANDARD",
            "PREMIUM",
            "ENTERPRISE"
        )
    )
    .withColumn(
        "dq_invalid_billing_cycle",
        ~F.col("billing_cycle_clean").isin(
            "MONTHLY",
            "QUARTERLY",
            "YEARLY"
        )
    )
    .withColumn(
        "dq_negative_fee",
        F.col("monthly_fee_cast").isNull()
        | (F.col("monthly_fee_cast") < 0)
    )
    .withColumn(
        "dq_invalid_date_range",
        F.col("start_date_cast").isNull()
        | (
            F.col("end_date_cast").isNotNull()
            & (F.col("end_date_cast") < F.col("start_date_cast"))
        )
    )
    .withColumn(
        "dq_critical_failure",
        F.col("dq_missing_subscription_id")
        | F.col("dq_missing_customer_id")
        | F.col("dq_customer_not_found")
        | F.col("dq_invalid_status")
        | F.col("dq_invalid_plan")
        | F.col("dq_invalid_billing_cycle")
        | F.col("dq_negative_fee")
        | F.col("dq_invalid_date_range")
    )
)

Count valid candidates

In [0]:
valid_candidate_count = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
    .count()
)

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

Valid candidates before dedup: 978,054


Calculate duplicate rows removed

In [0]:
duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Duplicate rows removed: 0


Perform reconciliation

In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows        : {bronze_count:,}")
print(f"Silver rows        : {silver_count:,}")
print(f"Rejected rows      : {reject_count:,}")
print(f"Duplicates removed : {duplicate_rows_removed:,}")
print(f"Accounted for      : {accounted_for:,}")
print(f"Difference         : {difference:,}")

Bronze rows        : 1,000,066
Silver rows        : 978,054
Rejected rows      : 22,012
Duplicates removed : 0
Accounted for      : 1,000,066
Difference         : 0


Create the reconciliation result

In [0]:
result = [
    {
        "entity": "subscriptions",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
1000066,1000066,0,0,subscriptions,22012,978054,PASS


Analyze rejection reasons

In [0]:
(
    reject_df
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+--------------------------------------------------------------------------------+-----+
|rejection_reason                                                                |count|
+--------------------------------------------------------------------------------+-----+
|CUSTOMER_NOT_FOUND                                                              |15952|
|INVALID_SUBSCRIPTION_STATUS                                                     |2946 |
|INVALID_DATE_RANGE                                                              |1994 |
|INVALID_MONTHLY_FEE                                                             |997  |
|MISSING_CUSTOMER_ID; CUSTOMER_NOT_FOUND; INVALID_MONTHLY_FEE; INVALID_DATE_RANGE|66   |
|CUSTOMER_NOT_FOUND; INVALID_SUBSCRIPTION_STATUS                                 |48   |
|INVALID_SUBSCRIPTION_STATUS; INVALID_DATE_RANGE                                 |6    |
|INVALID_SUBSCRIPTION_STATUS; INVALID_MONTHLY_FEE                                |3    |
+--------------------

Measure warnings in Silver

In [0]:
warning_summary = (
    silver_df
    .select(
        F.sum(
            F.when(
                F.col(
                    "dq_warning_cancelled_without_end_date"
                ) == True,
                1
            ).otherwise(0)
        ).alias(
            "cancelled_without_end_date"
        ),

        F.sum(
            F.when(
                F.col(
                    "dq_warning_expired_without_end_date"
                ) == True,
                1
            ).otherwise(0)
        ).alias(
            "expired_without_end_date"
        )
    )
)

display(warning_summary)

cancelled_without_end_date,expired_without_end_date
0,0


Verify no duplicate subscription IDs remain

In [0]:
duplicate_keys_in_silver = (
    silver_df
    .groupBy("subscription_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate subscription IDs in Silver:",
    duplicate_keys_in_silver
)

Duplicate subscription IDs in Silver: 0


Verify customer referential integrity

In [0]:
orphan_subscriptions = (
    silver_df.alias("s")
    .join(
        spark.table(
            "customer360_dev.silver.customers"
        )
        .select("customer_id")
        .alias("c"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orphan subscriptions in Silver:",
    orphan_subscriptions
)

Orphan subscriptions in Silver: 0
